# Threat Hunting with MITRE ATT&CK

Use ATT&CK coverage and recent alert evidence to build hunt hypotheses.

In [ ]:
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import pandas as pd
from clario360 import Client

sdk = Client.from_env()
LOOKBACK_DAYS = int(__import__('os').environ.get('CLARIO360_HUNT_WINDOW_DAYS', '30'))
alerts = sdk.cyber.alerts.list(date_from=(datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).isoformat(), per_page=500)
df = alerts.to_dataframe()
if df.empty:
    raise RuntimeError('No recent alerts found for MITRE hunt preparation.')



## Identify observed techniques

The same approach can be expanded with external ATT&CK coverage metadata if your environment has it.

In [ ]:
mitre_df = df.dropna(subset=['mitre_technique_id'])
technique_counts = mitre_df.groupby(['mitre_technique_id', 'mitre_technique_name']).size().reset_index(name='count').sort_values('count', ascending=False)
top = technique_counts.head(15).sort_values('count')
plt.figure(figsize=(10, 6))
plt.barh(top['mitre_technique_name'], top['count'], color='darkred')
plt.title('Observed MITRE Techniques')
plt.xlabel('Count')
plt.ylabel('Technique')
plt.tight_layout()
plt.show()
technique_counts.head(15)


## Draft hunt hypotheses

Use the most common techniques to generate hypotheses and follow-up query ideas.

In [ ]:
hypotheses = [
    f"Investigate whether technique {row.mitre_technique_id} ({row.mitre_technique_name}) is spreading laterally beyond current detections."
    for _, row in technique_counts.head(5).iterrows()
]
pd.DataFrame({'hunt_hypothesis': hypotheses})